# VarejoMix - dataset de recompra

Execute com **Run > Run All Cells** no JupyterLab. O PostgreSQL deve estar ativo. Nao ha substituicao silenciosa por dados offline.

## 1. Premissas

Corte: **2025-10-01 UTC**. Features em [corte-365 dias, corte); alvo em [corte, corte+90 dias). Somente vendas confirmadas com pagamento aprovado integral. A base sintetica possui atividade desde dezembro/2024. Status e categorias refletem o snapshot, sem historico de alteracoes; a separacao por data nao substitui um snapshot historico completo.

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql/08_features.sql").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro da pasta VarejoMix.")
sys.path.insert(0, str(ROOT / "scripts"))
from gerar_dataset import extract, segment, dictionary, export
import pandas as pd
pd.set_option("display.max_columns", 20)
print("Projeto VarejoMix  localizado.")

Projeto VarejoMix  localizado.


## 2. Consulta e extracao

A consulta separa pedidos e categorias antes de agregar, evitando duplicar valores pelo numero de itens. A funcao extract usa create_engine e pandas.read_sql, em scripts/gerar_dataset.py.

In [2]:
print((ROOT / "sql/08_features.sql").read_text(encoding="utf-8"))

-- Consulta parametrizada para SQLAlchemy/pandas: parametro :cutoff (UTC).
-- Regra de venda confirmada centralizada em analytics.vw_valid_orders.
WITH eligible_orders AS (
    SELECT order_id, customer_id, order_ts, channel, total_amount
    FROM analytics.vw_valid_orders
    WHERE order_ts >= CAST(:cutoff AS timestamptz) - INTERVAL '365 days'
      AND order_ts < CAST(:cutoff AS timestamptz)
), order_features AS (
    SELECT customer_id,
        ((CAST(:cutoff AS timestamptz) AT TIME ZONE 'UTC')::date - MAX(order_ts AT TIME ZONE 'UTC')::date)::int AS recency_days,
        COUNT(*)::int AS frequency_365d,
        ROUND(SUM(total_amount),2) AS monetary_365d,
        ROUND(AVG(total_amount),2) AS average_ticket,
        ROUND(AVG((channel='ECOMMERCE')::int),4) AS ecommerce_share
    FROM eligible_orders GROUP BY customer_id
), category_features AS (
    SELECT o.customer_id,COUNT(DISTINCT p.category)::int AS distinct_categories
    FROM eligible_orders o
    JOIN oltp.order_items i USIN

In [3]:
df, contexto = extract()
print("Extracao:", contexto["extraction"])
print("Ambiente:", contexto["validation_context"])
print("Servidor:", contexto["server_version"])
print("Corte:", contexto["cutoff_utc"])
print("Shape inicial:", df.shape)
display(df.head())

Extracao: PostgreSQL via SQLAlchemy/pandas.read_sql
Ambiente: Execucao local do usuario
Servidor: PostgreSQL 16.15 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit
Corte: 2025-10-01T00:00:00+00:00
Shape inicial: (162, 8)


,customer_id,recency_days,frequency_365d,monetary_365d,average_ticket,ecommerce_share,distinct_categories,repurchase_90d
0,1,2,310,180209.02,581.32,0.0903,5,1
1,2,17,176,103151.75,586.09,0.1932,5,0
2,3,1,118,69923.77,592.57,0.8051,5,0
3,5,3,71,37435.36,527.26,0.1972,5,1
4,6,1,48,28058.97,584.56,0.5000,5,1


## 3. Qualidade

Grao: um cliente elegivel por data de corte. Clientes sem compras historicas nao sao incluidos.

In [4]:
print("Duplicidade de cliente:", int(df.customer_id.duplicated().sum()))
display(df.isna().sum().to_frame("nulos"))
assert df.customer_id.is_unique
assert not df.isna().any().any()
assert df.repurchase_90d.isin([0, 1]).all()
display(df.describe().T)

Duplicidade de cliente: 0


,nulos
customer_id,0
recency_days,0
frequency_365d,0
monetary_365d,0
average_ticket,0
ecommerce_share,0
distinct_categories,0
repurchase_90d,0


,count,mean,std,min,25%,50%,75%,max
customer_id,162.0,98.679012,56.847803,1.0,51.2500,100.5000,146.7500,200.00
recency_days,162.0,18.709877,17.305685,1.0,5.0000,14.0000,26.0000,92.00
frequency_365d,162.0,17.660494,29.719008,1.0,8.0000,11.0000,18.0000,310.00
monetary_365d,162.0,10113.421049,17332.183585,197.9,4191.5675,5990.8550,10036.6125,180209.02
average_ticket,162.0,567.010185,110.758974,197.9,507.9975,573.2750,624.9875,974.11
ecommerce_share,162.0,0.535640,0.321597,0.0,0.2443,0.5556,0.8281,1.00
distinct_categories,162.0,4.783951,0.656729,1.0,5.0000,5.0000,5.0000,5.00
repurchase_90d,162.0,0.746914,0.436128,0.0,0.2500,1.0000,1.0000,1.00


## 4. Segmentacao RFM

Escores de 1 a 3 a partir da posicao percentual. Empates recebem o mesmo escore. Menor recencia e maior frequencia/valor recebem escores maiores. Nao ha treinamento de modelo neste recorte.

In [5]:
df = segment(df)
display(df[["customer_id", "r_score", "f_score", "m_score", "rfm_score", "segment"]].head(10))
display(df["segment"].value_counts().to_frame("clientes"))

,customer_id,r_score,f_score,m_score,rfm_score,segment
0,1,3,3,3,9,Alto valor
1,2,2,3,3,8,Alto valor
2,3,3,3,3,9,Alto valor
3,5,3,3,3,9,Alto valor
4,6,3,3,3,9,Alto valor
5,7,1,3,3,7,Alto valor
6,8,3,3,3,9,Alto valor
7,9,3,3,3,9,Alto valor
8,10,3,3,3,9,Alto valor
9,11,3,3,3,9,Alto valor


,clientes
segment,
Alto valor,70
Potencial,58
Manutencao,34


## 5. Dicionario completo

In [6]:
display(dictionary())
assert set(df.columns) == set(dictionary()["campo"])

,campo,tipo,descricao
0,customer_id,inteiro,Identificador tecnico sintetico; nao e uma ano...
1,recency_days,inteiro,Dias UTC desde a ultima venda confirmada anter...
2,frequency_365d,inteiro,"Vendas confirmadas na janela [corte-365 dias, ..."
3,monetary_365d,decimal BRL,Soma dos totais liquidos na janela historica.
4,average_ticket,decimal BRL,Valor medio por venda confirmada na janela.
5,ecommerce_share,decimal 0 a 1,Proporcao de vendas confirmadas no e-commerce.
6,distinct_categories,inteiro,Categorias distintas nos itens da janela histo...
7,repurchase_90d,binario,"1 se existe venda confirmada em [corte, corte+..."
8,r_score,inteiro 1 a 3,Escore de recencia: menor recencia recebe maio...
9,f_score,inteiro 1 a 3,Escore da posicao percentual da frequencia; em...


## 6. Exportacao e auditoria

CSV e Parquet sao comparados; cada execucao preserva sua pasta, manifesto, hash, corte e versao do codigo. O banco registra a exportacao real em audit.pipeline_runs. Isso e versionamento de arquivos; nao e Delta Lake/time travel.

In [7]:
manifesto = export(df, contexto)
print("Shape final:", df.shape)
print("Nulos:", manifesto["nulls"])
print("Snapshot:", manifesto["snapshot_directory"])
print("Versao do dataset:", manifesto["dataset_version"])
display(df.repurchase_90d.value_counts().sort_index().to_frame("clientes"))
print("CONCLUIDO: dataset exportado e auditado.")

Shape final: (162, 13)
Nulos: 0
Snapshot: outputs/execucoes/9cdc9572-d4af-40e0-88e0-7bbcd0580b14
Versao do dataset: sha256:c2676febd98e7b65d4dab43dad348bc7bb1c38f3b50b0c2b4bfcf42b4cac2fea


,clientes
repurchase_90d,
0,41
1,121


CONCLUIDO: dataset exportado e auditado.


## 7. Limites e proxima etapa

A carga e sintetica e pequena. O pipeline nao implementa ingestao diaria agendada, CDC nem Delta Lake. O relatorio apresenta essas capacidades como evolucao. Antes de entregar, execute os testes no Docker e guarde as evidencias locais; complete a identificacao e os links do repositorio e do video.